In [2]:
import polars as pl

PATH = "../data/raw/flat/50m/multi_event.parquet"

lf = pl.scan_parquet(PATH)
lf.collect_schema()

Schema([('uid', UInt32),
        ('timestamp', UInt32),
        ('item_id', UInt32),
        ('is_organic', UInt8),
        ('played_ratio_pct', UInt16),
        ('track_length_seconds', UInt32),
        ('event_type',
         Enum(categories=['listen', 'dislike', 'like', 'undislike', 'unlike']))])

In [3]:
lf.head(5).collect()

uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds,event_type
u32,u32,u32,u8,u16,u32,enum
100,39420,8326270,0,100,170,"""listen"""
100,39420,1441281,0,100,105,"""listen"""
100,39625,286361,0,100,185,"""listen"""
100,40110,732449,0,100,240,"""listen"""
100,40360,3397170,0,46,130,"""listen"""


In [4]:
lf.select(pl.len()).collect()

len
u32
47790449


In [5]:
lf.group_by("event_type").agg(pl.len()).sort("len", descending=True).collect()

event_type,len
enum,u32
"""listen""",46467212
"""like""",881456
"""unlike""",312972
"""dislike""",107776
"""undislike""",21033


In [6]:
lf.select(
    pl.col("uid").n_unique().alias("users"),
    pl.col("item_id").n_unique().alias("items"),
    pl.col("item_id").max().alias("max_item_id"),
    (pl.col("timestamp").min() / 86400).alias("first_day"),
    (pl.col("timestamp").max() / 86400).alias("last_day"),
).collect()

users,items,max_item_id,first_day,last_day
u32,u32,u32,f64,f64
10000,934057,9390623,0.0,300.925926


In [7]:
lf.group_by("event_type").agg(
    pl.col("played_ratio_pct").null_count().alias("nulls"),
    pl.col("played_ratio_pct").min().alias("min"),
    pl.col("played_ratio_pct").max().alias("max"),
).collect()

event_type,nulls,min,max
enum,u32,u16,u16
"""listen""",0,0,159
"""like""",881456,null,null
"""dislike""",107776,null,null
"""unlike""",312972,null,null
"""undislike""",21033,null,null


In [9]:
%load_ext autoreload
%autoreload 2
from yambda_rec.data import load_events, listen_plus, split, TEST_START

lp = listen_plus(load_events())
train, test = split(lp, TEST_START)

print(train.select(pl.len(), pl.col("timestamp").max()).collect())
print(test.select(
    pl.len(),
    pl.col("timestamp").min().alias("ts_min"),
    pl.col("timestamp").max().alias("ts_max"),
    pl.col("uid").n_unique().alias("users"),
).collect())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
shape: (1, 2)
┌──────────┬───────────┐
│ len      ┆ timestamp │
│ ---      ┆ ---       │
│ u32      ┆ u32       │
╞══════════╪═══════════╡
│ 29280230 ┆ 25911795  │
└──────────┴───────────┘
shape: (1, 4)
┌────────┬──────────┬──────────┬───────┐
│ len    ┆ ts_min   ┆ ts_max   ┆ users │
│ ---    ┆ ---      ┆ ---      ┆ ---   │
│ u32    ┆ u32      ┆ u32      ┆ u32   │
╞════════╪══════════╪══════════╪═══════╡
│ 157695 ┆ 25913600 ┆ 25999995 ┆ 4599  │
└────────┴──────────┴──────────┴───────┘
